# 가상환경 자동 설정
이 노트북을 실행하면 `.venv` 가상환경 생성 → 패키지 설치 → Jupyter 커널 등록 → 커널 자동 변경까지 완료됩니다.

In [1]:
import subprocess, sys, os, json, pathlib

# === 설정 ===
VENV_NAME = ".venv"
KERNEL_NAME = "tensorflow-venv"
KERNEL_DISPLAY = "Python (TensorFlow)"
PACKAGES = [
    "tensorflow",
    "matplotlib",
    "numpy",
    "pandas",
    "scikit-learn",
    "ipykernel",
]

PROJECT_DIR = pathlib.Path.cwd()
VENV_DIR = PROJECT_DIR / VENV_NAME

if os.name == "nt":
    PYTHON = VENV_DIR / "Scripts" / "python.exe"
    PIP = VENV_DIR / "Scripts" / "pip.exe"
else:
    PYTHON = VENV_DIR / "bin" / "python"
    PIP = VENV_DIR / "bin" / "pip"

def run(cmd, desc=""):
    if desc:
        print(f"\n>>> {desc}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        # 마지막 10줄만 출력
        lines = result.stdout.strip().split("\n")
        for line in lines[-10:]:
            print(f"  {line}")
    if result.returncode != 0 and result.stderr.strip():
        print(f"  [ERROR] {result.stderr.strip()[:300]}")
    return result.returncode == 0

print(f"프로젝트 경로: {PROJECT_DIR}")
print(f"가상환경 경로: {VENV_DIR}")
print(f"설치 패키지: {', '.join(PACKAGES)}")

프로젝트 경로: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice
가상환경 경로: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
설치 패키지: tensorflow, matplotlib, numpy, pandas, scikit-learn, ipykernel


In [2]:
# 1. 가상환경 생성
if VENV_DIR.exists():
    print(f"기존 가상환경 발견: {VENV_DIR}")
else:
    print("가상환경 생성 중...")
    run([sys.executable, "-m", "venv", str(VENV_DIR)], "python -m venv")
    print("가상환경 생성 완료")

# Python 경로 확인
assert PYTHON.exists(), f"Python 실행 파일을 찾을 수 없습니다: {PYTHON}"
print(f"venv Python: {PYTHON}")

기존 가상환경 발견: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
venv Python: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv\Scripts\python.exe


In [3]:
# 2. pip 업그레이드 + 패키지 설치
run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip"], "pip 업그레이드")
run([str(PIP), "install"] + PACKAGES, f"패키지 설치: {', '.join(PACKAGES)}")
print("\n패키지 설치 완료")


>>> pip 업그레이드

>>> 패키지 설치: tensorflow, matplotlib, numpy, pandas, scikit-learn, ipykernel

패키지 설치 완료


In [4]:
# 3. Jupyter 커널 등록
run(
    [str(PYTHON), "-m", "ipykernel", "install", "--user",
     f"--name={KERNEL_NAME}", f"--display-name={KERNEL_DISPLAY}"],
    "Jupyter 커널 등록"
)
print(f"\n커널 '{KERNEL_DISPLAY}' ({KERNEL_NAME}) 등록 완료")


>>> Jupyter 커널 등록
  Installed kernelspec tensorflow-venv in C:\Users\PDG\AppData\Roaming\jupyter\kernels\tensorflow-venv

커널 'Python (TensorFlow)' (tensorflow-venv) 등록 완료


In [5]:
# 4. 현재 노트북의 커널을 자동 변경
#    노트북 파일의 metadata를 직접 수정합니다.

NOTEBOOKS = list(PROJECT_DIR.glob("*.ipynb"))

changed = []
for nb_path in NOTEBOOKS:
    if nb_path.name == "envset.ipynb":
        continue
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    kernel_info = nb.get("metadata", {}).get("kernelspec", {})
    if kernel_info.get("name") == KERNEL_NAME:
        print(f"  {nb_path.name} - 이미 설정됨, 건너뜀")
        continue

    nb.setdefault("metadata", {})["kernelspec"] = {
        "display_name": KERNEL_DISPLAY,
        "language": "python",
        "name": KERNEL_NAME
    }
    with open(nb_path, "w", encoding="utf-8") as f:
        json.dump(nb, f, ensure_ascii=False, indent=1)
    changed.append(nb_path.name)
    print(f"  {nb_path.name} -> 커널 변경 완료")

print(f"\n총 {len(changed)}개 노트북 커널 변경 완료")
if changed:
    print(f"변경된 파일: {', '.join(changed)}")
print("\n노트북을 다시 열면 새 커널이 적용됩니다.")

  envsetting.ipynb -> 커널 변경 완료

총 1개 노트북 커널 변경 완료
변경된 파일: envsetting.ipynb

노트북을 다시 열면 새 커널이 적용됩니다.


In [6]:
# 5. 설치 확인
print("=== 설치 확인 ===")
result = subprocess.run(
    [str(PIP), "list", "--format=columns"],
    capture_output=True, text=True
)
for pkg in PACKAGES:
    name = pkg.replace("-", "_").lower()
    for line in result.stdout.split("\n"):
        if name in line.lower().replace("-", "_"):
            print(f"  {line.strip()}")
            break
    else:
        print(f"  [!] {pkg} - 설치 안 됨")

print(f"\n=== 완료 ===")
print(f"가상환경: {VENV_DIR}")
print(f"커널: {KERNEL_DISPLAY}")
print("노트북을 닫고 다시 열면 가상환경 커널로 실행됩니다.")

=== 설치 확인 ===
  [!] tensorflow - 설치 안 됨
  [!] matplotlib - 설치 안 됨
  [!] numpy - 설치 안 됨
  [!] pandas - 설치 안 됨
  [!] scikit-learn - 설치 안 됨
  [!] ipykernel - 설치 안 됨

=== 완료 ===
가상환경: c:\Users\PDG\OneDrive\Desktop\PJ\CODE\Ai-practice\.venv
커널: Python (TensorFlow)
노트북을 닫고 다시 열면 가상환경 커널로 실행됩니다.
